<a href="https://www.kaggle.com/code/gpreda/transformer-models-for-pii-detection?scriptVersionId=295174590" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Introduction

In this notebook I will test few (transformers based) models for PII detection.

PII stands for **Personally Identifiable Information**.

### What it means?

PII is any information that can identify a specific person, either on its own or when combined with other data.

Common examples of PII:
* Direct identifiers
* Full name
* Home address
* Email address
* Phone number
* Social Security number
* Passport or driver’s license number

Indirect identifiers (still PII when combined):
* Date of birth
* IP address
* Device IDs
* Employment details
* Location data
* Online usernames (in many contexts)

### Why PII matters?

PII is important because:
* It is protected by privacy and data protection laws (e.g., GDPR, US privacy laws)
* Must be handled, stored, and shared carefully
* Exposure can lead to identity theft, fraud, or privacy violations


# Prerequisites

Import here some packages that will be used in common.

In [1]:
from pprint import pprint
from time import time

# GLiNER

We will use GLiNER PII model (`nvidia/gliner-pii`).

GLiNER-PII supports detection and redaction of sensitive information across regulated and enterprise scenarios.

* **Healthcare**: Redact PHI in clinical notes, reports, and medical documents.
* **Finance**: Identify account numbers, SSNs, and transaction details in banking and insurance documents.
* **Legal**: Protect client information in contracts, filings, and discovery materials.
* **Enterprise Data Governance**: Scan documents, emails, and data stores for sensitive information.
* **Data Privacy Compliance**: Support GDPR, HIPAA, and CCPA workflows across varied document types.
* **Cybersecurity**: Detect sensitive data in logs, security reports, and incident records.
* **Content Moderation**: Flag personal information in user-generated content.

We need to install `gliner` library, not included with kaggle pre-provisioned kernel.

In [2]:
!pip install -qq gliner==0.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 21.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00


In [3]:
from gliner import GLiNER

model_gliner = GLiNER.from_pretrained("nvidia/gliner-pii")

2026-01-31 18:21:07.807033: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769883668.231914      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769883668.359182      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769883669.475448      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769883669.475490      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769883669.475493      24 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/1.78G [00:00<?, ?B/s]

gliner_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Let's test initially with some generic text.
Note: for GLiNER we set also the targeted labels.

In [4]:
text = "Hi support, I can't log in! My account username is 'johndoe88'. Every time I try, it says 'invalid credentials'. Please reset my password. You can reach me at (555) 123-4567 or johnd@example.com"
labels = ["email", "phone_number", "user_name"]

We create a test function.

In [5]:
def test_gliner(text, labels, model):
    start_time = time()
    entities = model.predict_entities(text, labels, threshold=0.5)
    end_time = time()
    print(f"Total time: {round(end_time-start_time, 2)} sec.")
    return entities

We apply it to one text example containing PII.

In [6]:
entities = test_gliner(text, labels, model_gliner)
pprint(entities)

Total time: 1.86 sec.
[{'end': 61, 'label': 'user_name', 'start': 52, 'text': 'johndoe88'},
 {'end': 173, 'label': 'phone_number', 'start': 159, 'text': '(555) 123-4567'},
 {'end': 194, 'label': 'email', 'start': 177, 'text': 'johnd@example.com'}]


We can also highlight the original text. Let's use for this an utility function.

In [7]:
import html

def highlight_pii(text, entities, colors=None):
    # simple per-label colors
    colors = colors or {
        "email": "#b3e5fc",        # light blue
        "phone_number": "#c8e6c9", # light green
        "user_name": "#fff9c4",    # light yellow
    }

    # sort by start index
    ents = sorted(entities, key=lambda e: (e["start"], -(e["end"] - e["start"])))

    out = []
    last = 0

    for e in ents:
        s, t = int(e["start"]), int(e["end"])
        label = str(e["label"])
        if s < last:   # skip overlaps in this simple version
            continue

        out.append(html.escape(text[last:s]))

        bg = colors.get(label, "#e0e0e0")
        chunk = html.escape(text[s:t])

        out.append(
            f"<span style='background:{bg}; padding:2px 4px; border-radius:4px;'>"
            f"{chunk}"
            f"</span>"
            f"<sup style='margin-left:4px; font-size:0.75em; opacity:0.75;'>[{html.escape(label)}]</sup>"
        )

        last = t

    out.append(html.escape(text[last:]))

    # keep line breaks
    return "<div style='white-space: pre-wrap; line-height:1.5; font-size:14px;'>" + "".join(out) + "</div>"


In [8]:
highlight_pii(text, entities)

"<div style='white-space: pre-wrap; line-height:1.5; font-size:14px;'>Hi support, I can&#x27;t log in! My account username is &#x27;<span style='background:#fff9c4; padding:2px 4px; border-radius:4px;'>johndoe88</span><sup style='margin-left:4px; font-size:0.75em; opacity:0.75;'>[user_name]</sup>&#x27;. Every time I try, it says &#x27;invalid credentials&#x27;. Please reset my password. You can reach me at <span style='background:#c8e6c9; padding:2px 4px; border-radius:4px;'>(555) 123-4567</span><sup style='margin-left:4px; font-size:0.75em; opacity:0.75;'>[phone_number]</sup> or <span style='background:#b3e5fc; padding:2px 4px; border-radius:4px;'>johnd@example.com</span><sup style='margin-left:4px; font-size:0.75em; opacity:0.75;'>[email]</sup></div>"

Let's define a set of tests. We will reuse them with the other models as well.

In [9]:
texts = ["Hi support, I can't log in! My account username is 'johndoe88'. Every time I try, it says 'invalid credentials'.\
        Please reset my password. You can reach me at (555) 123-4567 or johnd@example.com",
         "My name is Soong Hil Num and I am a British citizen. My phone number is 094123456 and my email address is soong@adobe.com",
        "Last time Lang Lang Li visited us was on board of Fae Li from Tokyo",
        "Nakamura-san did one last attempt to learn the Chinese filmography from Anne Admiral and use its input to talk with Yound Li",
        "Material science expert Li Young Sun met our specialist Franie Kuang to evaluate the feasibility of the approach.",
        "This text does not contain PII information and therefore should not be marked as so."]

In [10]:
for text in texts:
    entities = test_gliner(text, labels, model_gliner)
    pprint(entities)

Total time: 0.76 sec.
[{'end': 61, 'label': 'user_name', 'start': 52, 'text': 'johndoe88'},
 {'end': 180, 'label': 'phone_number', 'start': 166, 'text': '(555) 123-4567'},
 {'end': 201, 'label': 'email', 'start': 184, 'text': 'johnd@example.com'}]
Total time: 0.66 sec.
[{'end': 24, 'label': 'user_name', 'start': 11, 'text': 'Soong Hil Num'},
 {'end': 81, 'label': 'phone_number', 'start': 72, 'text': '094123456'},
 {'end': 121, 'label': 'email', 'start': 106, 'text': 'soong@adobe.com'}]
Total time: 0.62 sec.
[{'end': 22, 'label': 'user_name', 'start': 10, 'text': 'Lang Lang Li'}]
Total time: 0.65 sec.
[{'end': 12, 'label': 'user_name', 'start': 0, 'text': 'Nakamura-san'},
 {'end': 84, 'label': 'email', 'start': 72, 'text': 'Anne Admiral'}]
Total time: 0.62 sec.
[{'end': 36, 'label': 'user_name', 'start': 24, 'text': 'Li Young Sun'},
 {'end': 68, 'label': 'user_name', 'start': 56, 'text': 'Franie Kuang'}]
Total time: 0.61 sec.
[]


# BERT PII detection


* **Base Model**: distilbert-base-uncased
* **Task**: Token Classification (Named Entity Recognition)
* **Languages**: English
* **License**: MIT
* **Fine-tuned on**: AI4Privacy PII-42k dataset

### Supported PII Entity Types

This model can detect 56 different types of PII entities including:
* **Personal Information**: FIRSTNAME, LASTNAME, MIDDLENAME, EMAIL, PHONENUMBER, USERNAME, DATE, TIME, DOB, AGE
* **Address Information**: STREET, CITY, STATE, COUNTY, ZIPCODE, BUILDINGNUMBER, SECONDARYADDRESS
* **Financial Information**: CREDITCARDNUMBER, CREDITCARDISSUER, CREDITCARDCVV, ACCOUNTNAME,, ACCOUNTNUMBER, IBAN, BIC, AMOUNT, CURRENCY, CURRENCYCODE, CURRENCYSYMBOL
* **Identification**: SSN, PIN, PASSWORD, IP, IPV4, IPV6, MAC, ETHEREUMADDRESS, BITCOINADDRESS, LITECOINADDRESS
* **Professional Information**: JOBTITLE, JOBTYPE, JOBAREA, COMPANYNAME

In [11]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

# Load model and tokenizer
model_name = "SoelMgd/bert-pii-detection"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_bert_pii = AutoModelForTokenClassification.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

In [12]:
# Create NER pipeline
bert_ner_pii_pipeline = pipeline(
    "ner", 
    model=model_bert_pii, 
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

In [13]:
def test_bert_pII(text, ner_pipeline):
    start_time = time()
    entities = ner_pipeline(text)
    end_time = time()
    print(f"Total time: {round(end_time-start_time, 2)} sec.")
    return entities

In [14]:
entities = test_bert_pII(texts[0], bert_ner_pii_pipeline)
pprint(entities)

Total time: 0.21 sec.
[{'end': 59,
  'entity_group': 'USERNAME',
  'score': np.float32(0.35023725),
  'start': 56,
  'word': '##doe'},
 {'end': 180,
  'entity_group': 'PHONENUMBER',
  'score': np.float32(0.9891038),
  'start': 166,
  'word': '( 555 ) 123 - 4567'},
 {'end': 201,
  'entity_group': 'EMAIL',
  'score': np.float32(0.97653633),
  'start': 184,
  'word': 'johnd @ example. com'}]


In [15]:
for text in texts:
    entities = test_bert_pII(text, bert_ner_pii_pipeline)
    pprint(entities)

Total time: 0.01 sec.
[{'end': 59,
  'entity_group': 'USERNAME',
  'score': np.float32(0.35023725),
  'start': 56,
  'word': '##doe'},
 {'end': 180,
  'entity_group': 'PHONENUMBER',
  'score': np.float32(0.9891038),
  'start': 166,
  'word': '( 555 ) 123 - 4567'},
 {'end': 201,
  'entity_group': 'EMAIL',
  'score': np.float32(0.97653633),
  'start': 184,
  'word': 'johnd @ example. com'}]
Total time: 0.01 sec.
[{'end': 16,
  'entity_group': 'FIRSTNAME',
  'score': np.float32(0.77536356),
  'start': 11,
  'word': 'soong'},
 {'end': 24,
  'entity_group': 'LASTNAME',
  'score': np.float32(0.6947961),
  'start': 17,
  'word': 'hil num'},
 {'end': 81,
  'entity_group': 'PHONENUMBER',
  'score': np.float32(0.9559371),
  'start': 72,
  'word': '094123456'},
 {'end': 121,
  'entity_group': 'EMAIL',
  'score': np.float32(0.99195045),
  'start': 106,
  'word': 'soong @ adobe. com'}]
Total time: 0.01 sec.
[{'end': 14,
  'entity_group': 'FIRSTNAME',
  'score': np.float32(0.73828113),
  'start': 10

# BERT Small PII Detection

We test now the `gravitee-io/bert-small-pii-detection` model. It is a more compact model compared with the previous one, and is also based on BERT.

**Label set**: AGE, COORDINATE, CREDIT_CARD, DATE_TIME, EMAIL_ADDRESS, FINANCIAL, IBAN_CODE, IMEI,
IP_ADDRESS, LOCATION, MAC_ADDRESS, NRP, ORGANIZATION, PASSWORD, PERSON, PHONE_NUMBER,
TITLE, URL, US_BANK_NUMBER, US_DRIVER_LICENSE, US_ITIN, US_LICENSE_PLATE, US_PASSPORT, US_SSN

In [16]:
# Load model and tokenizer
model_name_s = "gravitee-io/bert-small-pii-detection"
tokenizer_s = AutoTokenizer.from_pretrained(model_name_s)
model_bert_small_pii = AutoModelForTokenClassification.from_pretrained(model_name_s)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [17]:
# Create NER pipeline
bert_small_ner_pii_pipeline = pipeline(
    "ner", 
    model=model_bert_small_pii, 
    tokenizer=tokenizer_s,
    aggregation_strategy="simple"
)

We can reuse the previous test function.

In [18]:
entities = test_bert_pII(texts[0], bert_small_ner_pii_pipeline)
pprint(entities)

Total time: 0.01 sec.
[{'end': 58,
  'entity_group': 'PERSON',
  'score': np.float32(0.5632639),
  'start': 52,
  'word': 'johndo'},
 {'end': 61,
  'entity_group': 'PASSWORD',
  'score': np.float32(0.80122596),
  'start': 58,
  'word': '##e88'},
 {'end': 180,
  'entity_group': 'PHONE_NUMBER',
  'score': np.float32(0.990106),
  'start': 166,
  'word': '( 555 ) 123 - 4567'},
 {'end': 201,
  'entity_group': 'EMAIL_ADDRESS',
  'score': np.float32(0.9923265),
  'start': 184,
  'word': 'johnd @ example. com'}]


In [19]:
for text in texts:
    entities = test_bert_pII(text, bert_small_ner_pii_pipeline)
    pprint(entities)

Total time: 0.01 sec.
[{'end': 58,
  'entity_group': 'PERSON',
  'score': np.float32(0.5632639),
  'start': 52,
  'word': 'johndo'},
 {'end': 61,
  'entity_group': 'PASSWORD',
  'score': np.float32(0.80122596),
  'start': 58,
  'word': '##e88'},
 {'end': 180,
  'entity_group': 'PHONE_NUMBER',
  'score': np.float32(0.990106),
  'start': 166,
  'word': '( 555 ) 123 - 4567'},
 {'end': 201,
  'entity_group': 'EMAIL_ADDRESS',
  'score': np.float32(0.9923265),
  'start': 184,
  'word': 'johnd @ example. com'}]
Total time: 0.01 sec.
[{'end': 24,
  'entity_group': 'PERSON',
  'score': np.float32(0.8703625),
  'start': 11,
  'word': 'soong hil num'},
 {'end': 81,
  'entity_group': 'PHONE_NUMBER',
  'score': np.float32(0.9581439),
  'start': 72,
  'word': '094123456'},
 {'end': 121,
  'entity_group': 'EMAIL_ADDRESS',
  'score': np.float32(0.9930496),
  'start': 106,
  'word': 'soong @ adobe. com'}]
Total time: 0.0 sec.
[{'end': 22,
  'entity_group': 'PERSON',
  'score': np.float32(0.93251806),
 

# Adaptive Classifier

This can be used as a pre-classifier in the pipeline for PII detection so that a PII model will be run only on identified texts.

In [20]:
!pip install -qq adaptive_classifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00


In [21]:
from adaptive_classifier import AdaptiveClassifier

classifier = AdaptiveClassifier.from_pretrained("adaptive-classifier/pii-detection")

config.json: 0.00B [00:00, ?B/s]

examples.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.55M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [22]:
def test_adaptive_classifier(text, classifier):
    start_time = time()
    predictions = classifier.predict(text)
    end_time = time()
    print(f"Total time: {round(end_time-start_time, 2)} sec.")
    return predictions

In [23]:
predictions = test_adaptive_classifier(text[0], classifier)
pprint(predictions)

W0131 18:22:32.686000 24 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Total time: 7.7 sec.
[('contains_pii', 0.6506147716513732), ('no_pii', 0.3493852283486269)]


In [24]:
for text in texts:
    predictions = test_adaptive_classifier(text, classifier)
    pprint(predictions)

Total time: 0.03 sec.
[('contains_pii', 0.6547039900866941), ('no_pii', 0.34529600991330583)]
Total time: 0.03 sec.
[('contains_pii', 0.6564886194554499), ('no_pii', 0.34351138054455)]
Total time: 0.03 sec.
[('contains_pii', 0.6508705867546245), ('no_pii', 0.34912941324537544)]
Total time: 0.03 sec.
[('contains_pii', 0.6549280856877995), ('no_pii', 0.3450719143122005)]
Total time: 0.03 sec.
[('no_pii', 0.6479265367015546), ('contains_pii', 0.3520734632984453)]
Total time: 0.03 sec.
[('no_pii', 0.6534181878049526), ('contains_pii', 0.3465818121950474)]


Observation: not all the text containing PII are identified correctly. This creates a problem if we want to use the classifier to pre-select the targeted sub-texts for PII analysis.

## Final remarks

We reviewed two PII NER models and one PII classification model.  

GLiNER model is accurate and covers many useful categories. It performs well with all Asian names we tried.

BERT PII detection model, based on `distilbert-base-uncased` is very fast but has lower accuracy.

BERT Small PII detection is slightly faster than the previous, the accuracy is better.

One interesting note is that for `Nakamura-san` - which is a Japanese name (including the particle for respectful addressing `san` both BERT-based models are failing to recognize as a name, it identifies it as an organization (company name). GLiNER is recognizing it as a person name (`user_name`).

We can use the classifier model (even faster) to select first what texts should be subjected to a PII detection model and only apply those (slower) to the selected texts. The drawback will be that the classifier model might miss some of the texts with PII.

